# Step 4 Leontief and Scope 2/3 Trace

This notebook displays public-safe trace checkpoints from the actual Step 4 EMRIO run. It follows the calculation path in `replication/pipeline/04_scope23.jl` and `replication/lib/scope23_leontief.jl`: load the Step 3 aggregate object, construct `I - A`, compute the Leontief inverse, form the multiplier `e = fL`, apply the SUT-row zeroing used by the implementation, build `f_ener`, and decompose Scope 2+3 into Scope 2 and Scope 3.

The notebook reads only audited aggregate checkpoint CSVs under `public_data/checkpoints/`. It does not load raw inputs, JLD2 intermediates, firm-level rows, or reversible matrices.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

CHECKPOINT_DIR = Path("../public_data/checkpoints")
TRACE_FILES = {
    "loaded": "11_step4_loaded_objects.csv",
    "matrix": "12_step4_matrix_construction.csv",
    "trace": "13_step4_leontief_inverse_trace.csv",
    "checks": "14_step4_scope23_equation_checks.csv",
}

def read_checkpoint(key):
    return pd.read_csv(CHECKPOINT_DIR / TRACE_FILES[key])

loaded = read_checkpoint("loaded")
matrix = read_checkpoint("matrix")
trace = read_checkpoint("trace")
checks = read_checkpoint("checks")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 6)

def compact(df, columns=None):
    out = df.copy()
    if columns is not None:
        out = out.loc[:, columns]
    return out

def fmt_value(x):
    if pd.isna(x):
        return ""
    try:
        x = float(x)
    except Exception:
        return x
    if abs(x) >= 1e6 or (abs(x) > 0 and abs(x) < 1e-4):
        return f"{x:.3e}"
    if abs(x - round(x)) < 1e-9:
        return f"{x:.0f}"
    return f"{x:.6f}"

## 1. Loaded Step 3 Objects

Step 4 starts from the Step 3 aggregate object. The public trace reports object dimensions and orientation only; matrix cells and row metadata are not released.

In [2]:
display(
    compact(
        loaded,
        ["object", "rows", "cols", "length", "orientation", "source_step", "public_release_status"],
    ).style.format({"rows": fmt_value, "cols": fmt_value, "length": fmt_value})
)

,object,rows,cols,length,orientation,source_step,public_release_status
0,T_agg,30810,30810,,matrix,03_emrio_aggregate.jl,aggregate diagnostics only; object not released
1,f,,,30810,row_vector,03_emrio_aggregate.jl,aggregate diagnostics only; object not released
2,A,30810,30810,,matrix,03_emrio_aggregate.jl,aggregate diagnostics only; object not released
3,I_EMRIO,30810,30810,,matrix,03_emrio_aggregate.jl,aggregate diagnostics only; object not released
4,info_qd_agg,30809,29,,table,03_emrio_aggregate.jl,aggregate diagnostics only; object not released
5,info_q_agg,30809,24,,table,03_emrio_aggregate.jl,aggregate diagnostics only; object not released
6,g_agg,,,30810,vector,03_emrio_aggregate.jl,aggregate diagnostics only; object not released


## 2. Matrix Construction

The calculation uses `A` and `I_EMRIO` to construct `I - A`. For `T_agg` and `A`, the trace reports dimensions, rounded density, finite-value checks, and aggregate totals. It intentionally does not publish nonzero-value quantiles or individual entries.

In [3]:
display(
    compact(
        matrix,
        [
            "artifact", "rows", "cols", "rounded_density_pct", "finite_value_check",
            "sum_total", "diag_min", "diag_median", "diag_max", "note",
        ],
    ).style.format({
        "rows": fmt_value,
        "cols": fmt_value,
        "rounded_density_pct": "{:.4f}",
        "sum_total": fmt_value,
        "diag_min": fmt_value,
        "diag_median": fmt_value,
        "diag_max": fmt_value,
    })
)

,artifact,rows,cols,rounded_density_pct,finite_value_check,sum_total,diag_min,diag_median,diag_max,note
0,T_agg,30810,30810,16.3451,True,1.499e+08,,,,No matrix entries or nonzero-value quantiles released.
1,A,30810,30810,16.7118,True,22542.935783,,,,No matrix entries or nonzero-value quantiles released.
2,I_EMRIO,30810,30810,0.0032,True,30810,1,1,1,Identity matrix diagnostic.
3,I_minus_A,30810,30810,16.3482,True,8267.064217,0.355636,1,10.210593,Constructed as I_EMRIO - A; no matrix entries released.


## 3. Code Path Map

The trace rows correspond directly to the Step 4 equations and variables.

In [4]:
code_path = pd.DataFrame([
    {"stage": "construct_IA", "operation": "IA = I_EMRIO - A", "public output": "dimension, rounded activity, finite checks"},
    {"stage": "compute_inverse", "operation": "L = inv(IA)", "public output": "dimension and fixed-column residual check"},
    {"stage": "compute_multiplier_e_after_sut_zeroing", "operation": "e_g23 = f * L; e_g23[SUT rows] = 0", "public output": "aggregate vector summary after SUT zeroing"},
    {"stage": "build_f_ener", "operation": "f_ener = build_energy_intensity_vector(...) ", "public output": "aggregate vector summary"},
    {"stage": "propagate_scope23", "operation": "g_S23 = e_g23 * T_agg", "public output": "aggregate output summary"},
    {"stage": "extract_scope2", "operation": "g_S2 = f_ener' * T_agg[1:end-1, 1:end-1]", "public output": "aggregate output summary"},
    {"stage": "compute_scope3_residual", "operation": "g_S3 = g_S23 - g_S2", "public output": "equation check and aggregate output summary"},
])
display(code_path)

,stage,operation,public output
0,construct_IA,IA = I_EMRIO - A,"dimension, rounded activity, finite checks"
1,compute_inverse,L = inv(IA),dimension and fixed-column residual check
2,compute_multiplier_e_after_sut_zeroing,e_g23 = f * L; e_g23[SUT rows] = 0,aggregate vector summary after SUT zeroing
3,build_f_ener,f_ener = build_energy_intensity_vector(...),aggregate vector summary
4,propagate_scope23,g_S23 = e_g23 * T_agg,aggregate output summary
5,extract_scope2,"g_S2 = f_ener' * T_agg[1:end-1, 1:end-1]",aggregate output summary
6,compute_scope3_residual,g_S3 = g_S23 - g_S2,equation check and aggregate output summary


## 4. Leontief Inverse and Multiplier Trace

The dense inverse and row-level vectors are not released. The public trace records their dimensions, fixed-column residual diagnostics, and aggregate summaries of derived vectors.

In [5]:
display(
    compact(
        trace,
        [
            "stage", "artifact", "rows", "cols", "length", "rounded_active_count",
            "total", "min", "p50", "p90", "p99", "max", "n_negative", "check_value", "note",
        ],
    ).style.format({
        "rows": fmt_value,
        "cols": fmt_value,
        "length": fmt_value,
        "rounded_active_count": fmt_value,
        "total": fmt_value,
        "min": fmt_value,
        "p50": fmt_value,
        "p90": fmt_value,
        "p99": fmt_value,
        "max": fmt_value,
        "n_negative": fmt_value,
        "check_value": fmt_value,
    })
)

,stage,artifact,rows,cols,length,rounded_active_count,total,min,p50,p90,p99,max,n_negative,check_value,note
0,construct_IA,I_minus_A,30810,30810,,1.552e+08,,,,,,,,,Sparse diagnostic only; matrix cells not released.
1,compute_inverse,Leontief_inverse,30810,30810,,,,,,,,0,,0,Dense inverse not released; max=maximum absolute residual and check_value=median absolute residual for fixed checked columns.
2,compute_multiplier_e_after_sut_zeroing,emission_multiplier_after_sut_zeroing,,,30810,,1.919e+07,-76569.626687,192.144133,1187.479332,5325.553247,1.648e+06,135,,"Computed as e = fL, then SUT rows are set to zero before propagation; row-level vector not released."
3,build_f_ener,energy_intensity_vector,,,30809,500,1.338e+06,0,0,0,499.227385,132882.198357,0,,Aggregate summary only; row-level vector not released.
4,propagate_scope23,scope2plus3_output,,,30809,,4.078e+10,-1.464e+08,5755.420119,1.496e+06,2.627e+07,3.839e+08,193,,Aggregate summary of Step 4 output.
5,extract_scope2,scope2_output,,,30809,,5.111e+09,-1.813e+07,343.155281,122361.146843,2.594e+06,1.604e+08,131,,Aggregate summary of Step 4 output.
6,compute_scope3_residual,scope3_output,,,30809,,3.567e+10,-1.446e+08,4576.446015,1.316e+06,2.346e+07,3.330e+08,255,,Aggregate summary of Step 4 output.


## 5. Scope 2/3 Equation Checks

These checks summarize the final Step 4 decomposition. The residual confirms that aggregate Scope 3 equals Scope 2+3 minus Scope 2 up to numerical precision. Emission totals are stored in the raw pipeline units; for comparison with the manuscript's aggregate uncertainty table, the first four totals are also shown in gigatonnes by dividing by 1e9.

In [6]:
checks_display = checks.copy()
checks_display["value_formatted"] = checks_display["value"].map(fmt_value)
scope_total_checks = {"scope1_total", "scope2_total", "scope23_total", "scope3_total"}
checks_display["value_gt"] = checks_display.apply(
    lambda r: f"{float(r['value']) / 1e9:.3f}" if r["check"] in scope_total_checks else "",
    axis=1,
)
display(checks_display[["check", "value_formatted", "value_gt", "equation", "note"]])

,check,value_formatted,value_gt,equation,note
0,scope1_total,2.489e+10,24.893,finite_sum(g_S1),Aggregate finite Scope 1 output from Step 4.
1,scope2_total,5.111e+09,5.111,finite_sum(g_S2),Aggregate finite Scope 2 output from Step 4.
2,scope23_total,4.078e+10,40.778,finite_sum(g_S23),Aggregate finite Scope 2+3 output from Step 4.
3,scope3_total,3.567e+10,35.667,finite_sum(g_S3),Aggregate finite Scope 3 residual from Step 4.
4,scope23_minus_scope2_minus_scope3_abs,7.629e-06,,abs(finite_sum(g_S23) - finite_sum(g_S2) - finite_sum(g_S3)),Numerical decomposition check.
5,n_negative_multiplier_e_g23,135,,count(e_g23 < 0),Multiplier-vector numerical diagnostic after SUT zeroing.
6,n_negative_scope3_total,255,,count(g_S3 < 0),Scope 3 output diagnostic.
7,inverse_residual_checked_columns,15,,fixed first/middle/last columns,Deterministic inverse check.
8,inverse_residual_max_abs,0,,max(abs((I-A)L_selected - I_selected)),Deterministic inverse check.
9,inverse_residual_median_abs,0,,median(abs((I-A)L_selected - I_selected)),Deterministic inverse check.


## 6. Minimal Consistency Assertions

The public trace should satisfy dimension consistency, inverse-check availability, finite equation-check values, and exact aggregate decomposition within numerical tolerance.

In [7]:
loaded_lookup = loaded.set_index("object")
assert int(loaded_lookup.loc["T_agg", "rows"]) == int(loaded_lookup.loc["A", "rows"])
assert int(loaded_lookup.loc["T_agg", "cols"]) == int(loaded_lookup.loc["A", "cols"])
assert int(loaded_lookup.loc["f", "length"]) == int(loaded_lookup.loc["A", "cols"])
assert int(loaded_lookup.loc["info_qd_agg", "rows"]) == int(loaded_lookup.loc["T_agg", "rows"]) - 1

check_lookup = checks.set_index("check")["value"]
assert float(check_lookup.loc["inverse_residual_checked_columns"]) == 15
assert float(check_lookup.loc["scope23_minus_scope2_minus_scope3_abs"]) < 1e-3
assert checks["value"].notna().all()

pd.DataFrame({
    "assertion": [
        "T_agg and A dimensions match",
        "f length matches A columns",
        "destination rows equal matrix dimension minus RoW row",
        "inverse residual check used 15 fixed columns",
        "Scope 3 aggregate residual is below tolerance",
        "equation-check values are finite",
    ],
    "status": ["pass"] * 6,
})

,assertion,status
0,T_agg and A dimensions match,pass
1,f length matches A columns,pass
2,destination rows equal matrix dimension minus RoW row,pass
3,inverse residual check used 15 fixed columns,pass
4,Scope 3 aggregate residual is below tolerance,pass
5,equation-check values are finite,pass


## 7. Public-Release Audit

This cell checks the loaded public trace tables for restricted identifiers, local absolute paths, invalid numeric values, and accidental release of dense-matrix or row-level-vector columns.

In [8]:
frames = {
    "loaded": loaded,
    "matrix": matrix,
    "trace": trace,
    "checks": checks,
}

restricted_tokens = [
    "FACT" + "SET",
    "ENT" + "ITY",
    "PRO" + "PER",
    "ACCOUNT" + "_NAME",
    "IS" + "IN",
]
path_tokens = ["/" + "mnt/", "/" + "home/"]
trace_column_tokens = [
    "t_Leon_full", "t_Leon", "e_g23", "f_ener", "T_agg_entry", "A_entry", "matrix_cell",
]

issues = []
for name, df in frames.items():
    joined_columns = " ".join(map(str, df.columns))
    joined_values = " ".join(df.astype(str).to_numpy().ravel())
    for token in restricted_tokens:
        if token in joined_columns or token in joined_values:
            issues.append((name, "restricted_identifier_token", token))
    for token in path_tokens:
        if token in joined_columns or token in joined_values:
            issues.append((name, "local_absolute_path", token))
    for token in trace_column_tokens:
        if token.lower() in joined_columns.lower():
            issues.append((name, "unsafe_trace_column", token))

if checks["value"].isna().any():
    issues.append(("checks", "missing_required_check_value", "value"))
if not matrix["finite_value_check"].all():
    issues.append(("matrix", "finite_value_check_failed", "finite_value_check"))

assert not issues, issues
pd.DataFrame({
    "audit": [
        "restricted identifier/path scan",
        "dense-matrix/vector column-name scan",
        "required numeric check scan",
    ],
    "status": ["pass", "pass", "pass"],
})

,audit,status
0,restricted identifier/path scan,pass
1,dense-matrix/vector column-name scan,pass
2,required numeric check scan,pass
